In [ ]:
import os
os.environ["DEEPFACE_BACKEND_ENGINE"] = "onnx"

import time
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

from retinaface import RetinaFace
from mtcnn import MTCNN

In [ ]:
data = pd.read_csv("C:/DATA/face-detection/training.csv")
data.head()

In [ ]:
data.info()

In [ ]:
# Thống kê số nhãn bị thiếu
missing = data.isnull().sum().sort_values(ascending=False)

display(missing[missing > 0])
print("Total images:", len(data))
print("Images with all labels:", data.dropna().shape[0])

In [ ]:
# Các landmark cần đánh giá
KEYPOINTS = [
    "left_eye_center",
    "right_eye_center",
    "nose_tip",
    "mouth_left_corner",
    "mouth_right_corner"
]

COORD_COLUMNS = [
    f"{point}_{axis}"
    for point in KEYPOINTS
    for axis in ("x", "y")
]

# Chỉ giữ ảnh có đủ nhãn cho cả 5 landmark
valid_data = data.dropna(subset=COORD_COLUMNS + ["Image"]).copy()

print("Valid images:", len(valid_data))
print("Excluded images:", len(data) - len(valid_data))

In [ ]:
# Chuyển dữ liệu pixel thành ảnh NumPy
def parse_image(pixel_string):
    pixels = np.fromstring(pixel_string, sep=" ", dtype=np.uint8)

    if pixels.size != 96 * 96:
        raise ValueError(f"Expected 9216 pixels, got {pixels.size}")

    return pixels.reshape(96, 96)


images = np.stack([
    parse_image(pixel_string)
    for pixel_string in valid_data["Image"]
])

# Ground truth: mỗi ảnh có 5 điểm, mỗi điểm gồm x và y
ground_truth = valid_data[COORD_COLUMNS].to_numpy(
    dtype=np.float32
).reshape(-1, 5, 2)

print("Images shape:", images.shape)
print("Ground truth shape:", ground_truth.shape)

In [ ]:
# Hiển thị một ảnh và 5 landmark ground truth
idx = 0

plt.figure(figsize=(5, 5))
plt.imshow(images[idx], cmap="gray")

for i, (x, y) in enumerate(ground_truth[idx]):
    plt.scatter(x, y, s=35, label=KEYPOINTS[i])
    plt.text(x + 1, y + 1, str(i), color="yellow", fontsize=10)

plt.xlim(0, 96)
plt.ylim(96, 0)
plt.title("Ground-truth facial landmarks")
plt.legend(fontsize=8)
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split

# Giữ nguyên index để truy xuất nhãn tương ứng
all_indices = np.arange(len(images))

_, val_indices = train_test_split(
    all_indices,
    test_size=0.2,
    random_state=42
)

# Giới hạn số ảnh để chạy thử ban đầu
MAX_VAL_IMAGES = 500
val_indices = val_indices[:MAX_VAL_IMAGES]

val_images = images[val_indices]
val_ground_truth = ground_truth[val_indices]

print("Validation images:", len(val_images))

# Khởi tạo mô hình

In [ ]:
class FaceLandmarkDetector:
    def __init__(self, model_path):
        base_options = mp.tasks.BaseOptions(
            model_asset_path=str(model_path)
        )

        options = mp.tasks.vision.FaceLandmarkerOptions(
            base_options=base_options,
            running_mode=mp.tasks.vision.RunningMode.IMAGE,
            num_faces=1,
            min_face_detection_confidence=0.5,
            min_face_presence_confidence=0.5,
            min_tracking_confidence=0.5
        )

        self.landmarker = (
            mp.tasks.vision.FaceLandmarker
            .create_from_options(options)
        )

        self.timestamp_ms = 0

    def process(self, frame):
        # OpenCV dùng BGR → chuyển sang RGB
        rgb_frame = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2RGB
        )

        # Chuyển ảnh sang MediaPipe Image
        image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=rgb_frame
        )

        result = self.landmarker.detect(image)

        return result.face_landmarks

    def close(self):
        self.landmarker.close()

In [ ]:
class RetinaFaceDetector:
    def __init__(self):
        # RetinaFace tự tải hoặc nạp weights qua thư viện
        pass

    def process(self, frame):
        # OpenCV BGR -> RGB
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # Phát hiện khuôn mặt và 5 facial landmarks
        result = RetinaFace.detect_faces(rgb_frame)

        if not isinstance(result, dict) or not result:
            return []

        # Chọn khuôn mặt có bounding box lớn nhất
        face = max(
            result.values(),
            key=lambda item: (
                item["facial_area"][2] - item["facial_area"][0]
            ) * (
                item["facial_area"][3] - item["facial_area"][1]
            )
        )

        return face.get("landmarks", {})

In [ ]:
class MTCNNDetector:
    def __init__(self):
        # Khởi tạo model một lần, không tạo lại ở mỗi ảnh
        self.detector = MTCNN()

    def process(self, frame):
        # OpenCV BGR -> RGB
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # Phát hiện khuôn mặt và 5 facial landmarks
        faces = self.detector.detect_faces(rgb_frame)

        if not faces:
            return []

        # Chọn khuôn mặt có bounding box lớn nhất
        face = max(
            faces,
            key=lambda item: item["box"][2] * item["box"][3]
        )

        return face.get("keypoints", {})

In [ ]:
TASK_PATH = "../face_landmarker.task"

mediapipe_detector = FaceLandmarkDetector(TASK_PATH)
retinaface_detector = RetinaFaceDetector()
mtcnn_detector = MTCNNDetector()

In [ ]:
# MediaPipe Face Landmarker đã được khởi tạo ở cell trước
# RetinaFace và MTCNN cũng cần import thành công

def prepare_image(gray):
    """Chuyển ảnh grayscale 96x96 thành RGB 192x192."""
    enlarged = cv2.resize(
        gray, (192, 192),
        interpolation=cv2.INTER_CUBIC
    )
    return cv2.cvtColor(enlarged, cv2.COLOR_GRAY2RGB)


def sort_by_x(points):
    """Sắp xếp landmark từ trái sang phải theo ảnh."""
    points = np.asarray(points, dtype=np.float32)
    return points[np.argsort(points[:, 0])]

# Benchmark mô hình

In [ ]:
def mediapipe_predict(rgb):
    """Ánh xạ Face Mesh thành 5 điểm trên ảnh gốc 96x96."""
    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=np.ascontiguousarray(rgb)
    )

    # Model đang chạy ở IMAGE mode nên gọi detect()
    result = mediapipe_detector.landmarker.detect(mp_image)

    if not result.face_landmarks:
        return None

    lm = result.face_landmarks[0]

    def point(index):
        return np.array(
            [lm[index].x * rgb.shape[1], lm[index].y * rgb.shape[0]],
            dtype=np.float32
        )

    # Ước lượng tâm mắt bằng trung điểm hai góc mắt
    eyes = sort_by_x([
        (point(33) + point(133)) / 2,
        (point(362) + point(263)) / 2
    ])

    # Chóp mũi và hai khóe miệng
    mouth = sort_by_x([point(61), point(291)])
    nose = point(1)

    # 192x192 -> 96x96
    return np.array(
        [eyes[0], eyes[1], nose, mouth[0], mouth[1]]
    ) / 2

In [ ]:
def retinaface_predict(rgb):
    bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
    result = RetinaFace.detect_faces(bgr, threshold=0.7)

    if not isinstance(result, dict) or not result:
        return None

    face = max(
        result.values(),
        key=lambda item: (
            item["facial_area"][2] - item["facial_area"][0]
        ) * (
            item["facial_area"][3] - item["facial_area"][1]
        )
    )

    lm = face["landmarks"]

    # Sắp xếp các điểm theo tọa độ x để thống nhất thứ tự
    eyes = sort_by_x([lm["left_eye"], lm["right_eye"]])
    mouth = sort_by_x([lm["mouth_left"], lm["mouth_right"]])

    points = np.array(
        [eyes[0], eyes[1], lm["nose"], mouth[0], mouth[1]],
        dtype=np.float32
    )

    return points / 2.0

In [ ]:
def mtcnn_predict(rgb):
    """Lấy 5 landmark từ MTCNN."""
    faces = mtcnn_detector.detector.detect_faces(rgb)

    if not faces:
        return None

    face = max(
        faces,
        key=lambda item: item["box"][2] * item["box"][3]
    )

    lm = face.get("keypoints", {})
    required = [
        "left_eye", "right_eye", "nose",
        "mouth_left", "mouth_right"
    ]

    if not all(key in lm for key in required):
        return None

    eyes = sort_by_x([lm["left_eye"], lm["right_eye"]])
    mouth = sort_by_x([lm["mouth_left"], lm["mouth_right"]])

    points = np.array(
        [eyes[0], eyes[1], lm["nose"], mouth[0], mouth[1]],
        dtype=np.float32
    )

    return points / 2

In [ ]:
rgb = prepare_image(val_images[0])

for name, predict_fn in {
    "MediaPipe": mediapipe_predict,
    "RetinaFace": retinaface_predict,
    "MTCNN": mtcnn_predict
}.items():
    try:
        pred = predict_fn(rgb)
        print(name, ":", None if pred is None else pred.shape)
    except Exception as e:
        print(name, "ERROR:", repr(e))

# Kết quả

In [ ]:
import time

predictors = {
    "MediaPipe": mediapipe_predict,
    "RetinaFace": retinaface_predict,
    "MTCNN": mtcnn_predict
}

benchmark_results = {}

for model_name, predict_fn in predictors.items():
    predictions = []
    elapsed_times = []
    errors = []

    print(f"Running {model_name}...")

    for i, gray in enumerate(val_images):
        rgb = prepare_image(gray)

        start = time.perf_counter()

        try:
            pred = predict_fn(rgb)
        except Exception as exc:
            pred = None
            errors.append((i, str(exc)))

        elapsed = time.perf_counter() - start
        elapsed_times.append(elapsed)

        predictions.append(pred)

        if (i + 1) % 50 == 0:
            print(f"  Processed {i + 1}/{len(val_images)}")

    benchmark_results[model_name] = {
        "predictions": predictions,
        "times": np.array(elapsed_times),
        "errors": errors
    }

    detected = sum(p is not None for p in predictions)

    print(
        f"{model_name}: {detected}/{len(val_images)} "
        f"faces detected; {len(errors)} errors"
    )

print("Inference finished.")

In [ ]:
from landmark_metrics import evaluate_benchmark
summary_df = pd.DataFrame(evaluate_benchmark(val_ground_truth, benchmark_results))
display(summary_df.sort_values("NME common (%)"))


In [ ]:
def visualize_predictions(index):
    gray = val_images[index]
    gt = val_ground_truth[index]

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))

    axes[0].imshow(gray, cmap="gray")
    axes[0].scatter(gt[:, 0], gt[:, 1], marker="x")
    axes[0].set_title("Ground truth")

    for ax, (model_name, result) in zip(
        axes[1:], benchmark_results.items()
    ):
        ax.imshow(gray, cmap="gray")

        pred = result["predictions"][index]

        if pred is not None:
            ax.scatter(pred[:, 0], pred[:, 1], marker="o")
        else:
            ax.text(
                0.5, 0.5, "No detection",
                transform=ax.transAxes,
                ha="center"
            )

        ax.set_title(model_name)

    for ax in axes:
        ax.set_xlim(0, 96)
        ax.set_ylim(96, 0)

    plt.tight_layout()
    plt.show()


visualize_predictions(0)